# Notebook 04b — Análisis topológico DE-9IM sobre AOI acotado (versión vigente)

Aplica los predicados topológicos DE-9IM enseñados en el Cap. 16 del curso y la Práctica 2 de vectores sobre los parches de manglar reproyectados a EPSG:9377. Adicionalmente, clasifica las 8 estaciones de muestreo por naturaleza espectral (manglar vs limnológica) y por asociación con el AOI mediante buffer de 2 km.

Esta es la **versión limpia** del análisis topológico. El notebook `04b_topologia.ipynb` original incluye también el análisis sobre AOI envolvente del baseline, que se conserva para trazabilidad.

**Insumos:**
- `data/processed/samgeo_acotado/manglar_*_9377.geojson`
- `data/raw/cgsm_aoi_acotado_9377.geojson`

**Productos:**
- `outputs/tables/parches_topologia_acotado.csv`
- `outputs/tables/estaciones_clasificadas.csv`
- `data/processed/samgeo_acotado/manglar_*_topologia.geojson`

In [ ]:
import sys; sys.path.insert(0, '../src')
from pathlib import Path
import geopandas as gpd
import pandas as pd
from python.utils import (
    area_ha_9377, parches_borde, parches_con_punto,
    estaciones_geodataframe, etiquetar_estaciones_aoi, EPSG_NACIONAL,
)

ROOT     = Path('..').resolve()
BASE     = ROOT / 'data' / 'processed' / 'samgeo_acotado'
AOI_PATH = ROOT / 'data' / 'raw' / 'cgsm_aoi_acotado_9377.geojson'
OUT_DIR  = ROOT / 'outputs' / 'tables'
OUT_DIR.mkdir(parents=True, exist_ok=True)

PERIODOS = ['degradacion', 'recuperacion', 'actual']
print(f'EPSG objetivo: {EPSG_NACIONAL}')
print(f'AOI: {AOI_PATH.exists()} {AOI_PATH}')

## 1. Carga del AOI acotado y de las 8 estaciones de muestreo

In [ ]:
aoi = gpd.read_file(AOI_PATH)
print(f'AOI: {len(aoi)} poligono(s), CRS={aoi.crs}')

estaciones = estaciones_geodataframe()
print(f'Estaciones: {len(estaciones)}')
print(estaciones[['estacion','fuente','naturaleza']].to_string(index=False))

## 2. Predicados topológicos DE-9IM por período

Para cada uno de los tres GeoJSON reproyectados a 9377 se aplican dos predicados: `intersects` entre parches y la frontera del AOI con tolerancia de 30 m (identifica parches truncados por el límite del área de estudio), y `contains` entre parches y los 8 puntos de muestreo (validación cualitativa de cobertura).

In [ ]:
filas = []
for nombre in PERIODOS:
    p = BASE / f'manglar_{nombre}_9377.geojson'
    if not p.exists():
        print(f'  no encontrado: {p}'); continue
    gdf = area_ha_9377(gpd.read_file(p))
    gdf = gdf[(gdf['area_ha'] >= 1.0) & (gdf['area_ha'] < 5000.0)].copy()
    
    gdf = parches_borde(gdf, aoi, tolerancia_m=30.0)
    gdf = parches_con_punto(gdf, estaciones.to_crs(EPSG_NACIONAL))
    
    n_total    = len(gdf)
    n_borde    = int(gdf['borde'].sum())
    n_estacion = int(gdf['con_estacion'].sum())
    area_borde = float(gdf.loc[gdf['borde'], 'area_ha'].sum())
    
    filas.append({
        'periodo': nombre,
        'parches': n_total,
        'parches_borde': n_borde,
        'pct_borde': round(100 * n_borde / max(n_total, 1), 1),
        'area_ha_borde': round(area_borde, 1),
        'parches_con_estacion': n_estacion,
        'estaciones_dentro': ', '.join(sorted(set(
            s for sub in gdf['estacion_dentro'].dropna() for s in sub.split(', ')
        ))),
    })
    
    out = BASE / f'manglar_{nombre}_topologia.geojson'
    gdf[['area_ha','perim_m','borde','con_estacion','estacion_dentro','geometry']].to_file(
        out, driver='GeoJSON')
    print(f'  {nombre}: {n_total} parches, {n_borde} en borde, {n_estacion} con estacion')

df_topo = pd.DataFrame(filas)
df_topo.to_csv(OUT_DIR / 'parches_topologia_acotado.csv', index=False)
print('\n=== Tabla topologia ===')
print(df_topo.to_string(index=False))
print(f'\nGuardado: outputs/tables/parches_topologia_acotado.csv')

## 3. Clasificación de estaciones por naturaleza espectral y asociación al AOI

In [ ]:
est = etiquetar_estaciones_aoi(estaciones, aoi, buffer_m=2000.0)
df_est = pd.DataFrame(est.drop(columns='geometry'))
df_est['dist_a_aoi_m'] = df_est['dist_a_aoi_m'].round(0).astype(int)

# Cruzar con NDVI mediano historico
ndvi_path = OUT_DIR / 'serie_temporal_ndvi_definitiva.csv'
if ndvi_path.exists():
    ndvi = pd.read_csv(ndvi_path)
    ndvi_mediano = ndvi.groupby('subzona')['ndvi'].median().round(3).to_dict()
    df_est['ndvi_mediano'] = df_est['estacion'].map(ndvi_mediano)

df_est = df_est.sort_values(['naturaleza','fuente','estacion']).reset_index(drop=True)
df_est.to_csv(OUT_DIR / 'estaciones_clasificadas.csv', index=False)
print(df_est.to_string(index=False))
print('\nGuardado: outputs/tables/estaciones_clasificadas.csv')